# LeetCode #3 — Longest Substring Without Repeating Characters: nhập môn Sliding Window

**Mức độ:** Medium · **Chủ đề:** String, Hash Set / Hash Map, Sliding Window

Skeleton trên LeetCode:

```python
class Solution:
    def lengthOfLongestSubstring(self, s: str) -> int:
```

Mục tiêu của notebook là giúp bạn hiểu **kỹ thuật cửa sổ trượt (Sliding Window)** — một trong những kỹ thuật quan trọng nhất cho các bài "tìm đoạn con dài nhất / ngắn nhất thỏa điều kiện" trên chuỗi và mảng.

---

## Sau notebook này, bạn sẽ nắm được

1. Phân biệt **substring** (chuỗi con liên tiếp) và **subsequence** (dãy con).
2. Cách dùng slicing `s[i:j]` và `set` để kiểm tra trùng lặp.
3. Vì sao cách thử mọi chuỗi con chạy chậm, và chỗ nào đang làm lại việc thừa.
4. Ý tưởng cửa sổ trượt với hai con trỏ `left`, `right`.
5. Cách tối ưu bằng `dict` lưu vị trí xuất hiện gần nhất, và vì sao cần `max(...)`.
6. Mẹo nhận diện bài Sliding Window.
7. Bài tập vận dụng từ dễ đến nâng cao.

# 1. Bài toán là gì?

Cho một chuỗi `s`. Hãy tìm **độ dài** của chuỗi con (substring) dài nhất **không có ký tự nào lặp lại**.

**Constraints (theo đề):**

- `0 <= s.length <= 5 * 10^4`
- `s` gồm chữ cái tiếng Anh, chữ số, ký hiệu và dấu cách.

---

## Ví dụ 1

```text
s = "abcabcbb"  ->  3
```

Chuỗi con dài nhất không lặp là `"abc"`, độ dài 3.

## Ví dụ 2

```text
s = "bbbbb"  ->  1
```

Chỉ có thể lấy `"b"`.

## Ví dụ 3

```text
s = "pwwkew"  ->  3
```

Đáp án là `"wke"`. Lưu ý: `"pwke"` **không phải** substring vì các ký tự không liền nhau trong `s` — nó là một *subsequence*.

---

## Điều quan trọng

Đề yêu cầu trả về **độ dài** (một số nguyên), không phải chính chuỗi con.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Bảng băm (hash table) — `dict` và `set`
2. Cửa sổ trượt (sliding window)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Bảng băm (hash table) — `dict` và `set`

### Bảng băm là gì?

Bảng băm lưu các cặp **khóa → giá trị** sao cho **tra cứu, thêm, xóa theo khóa** chỉ mất trung bình `O(1)`.

Ý tưởng: một **hàm băm** biến khóa thành một con số; con số đó chỉ ra **ngăn** (bucket) trong một mảng nơi cất giá trị. Thay vì dò lần lượt cả danh sách (`O(n)`), ta tính hàm băm rồi nhảy thẳng tới ngăn.

```text
khóa "cam"  --hash--> 7  ──►  ngăn[7] = ("cam", 3)
khóa "táo"  --hash--> 2  ──►  ngăn[2] = ("táo", 5)
khóa "lê"   --hash--> 7  ──►  ngăn[7] đã có "cam" → VA CHẠM, Python tự xử lý (dò ngăn khác)
```

Python có sẵn hai kiểu dựa trên bảng băm:

| Kiểu | Lưu gì | Ví dụ |
|---|---|---|
| `dict` | khóa → giá trị | `{"cam": 3, "táo": 5}` |
| `set` | chỉ khóa (tập hợp, không trùng) | `{1, 4, 9}` |

**Khóa phải "hashable"** (bất biến): `int`, `str`, `tuple`, `frozenset` dùng được; `list`, `dict`, `set` thì **không**.

In [1]:
# --- dict: khóa -> giá trị ---
gia = {"cam": 3, "táo": 5}
gia["lê"] = 4                     # thêm
gia["cam"] = 6                    # cập nhật
print("gia['táo'] =", gia["táo"])
print("'xoài' có trong dict?", "xoài" in gia)        # kiểm tra khóa: O(1)
print("get có giá trị mặc định:", gia.get("xoài", 0))
del gia["lê"]                     # xóa
for k, v in gia.items():          # duyệt cặp khóa-giá trị
    print(" ", k, "->", v)

# --- set: tập hợp không trùng ---
s = {3, 1, 3, 2}
s.add(5); s.discard(1)
print("set:", s, "| 2 in s:", 2 in s)
print("giao:", {1, 2, 3} & {2, 3, 4}, "| hợp:", {1, 2} | {5}, "| hiệu:", {1, 2, 3} - {2})

gia['táo'] = 5
'xoài' có trong dict? False
get có giá trị mặc định: 0
  cam -> 6
  táo -> 5
set: {2, 3, 5} | 2 in s: True
giao: {2, 3} | hợp: {1, 2, 5} | hiệu: {1, 3}


### Vì sao nhanh? So sánh `list` và `set` khi kiểm tra phần tử

`x in list` phải so sánh lần lượt từng phần tử (`O(n)`); `x in set` chỉ tính hàm băm (`O(1)` trung bình).

In [2]:
import time as _time_hash

_data = list(range(200_000))
_as_set = set(_data)
_t0 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _data
_t1 = _time_hash.perf_counter()
for _x in range(199_000, 199_100):
    _ = _x in _as_set
_t2 = _time_hash.perf_counter()
print(f"100 lần 'in' trên list: {(_t1 - _t0) * 1000:.2f} ms | trên set: {(_t2 - _t1) * 1000:.3f} ms")

100 lần 'in' trên list: 546.72 ms | trên set: 0.089 ms


### Bốn mẫu dùng bảng băm hay gặp nhất

1. **Đếm tần suất** — `Counter` hoặc `d[x] = d.get(x, 0) + 1`.
2. **Nhóm theo khóa** — `defaultdict(list)`, khóa là "đặc trưng chung" của nhóm.
3. **Ghi nhớ thứ đã thấy** — lưu giá trị → vị trí để trả lời "đã gặp phần bù chưa?" trong `O(1)` (kiểu Two Sum).
4. **Đánh dấu đã thăm** — `visited = set()` khi duyệt đồ thị / lưới.

In [3]:
from collections import Counter, defaultdict

# 1) Đếm
print("Counter:", Counter("banana"))

# 2) Nhóm theo khóa: nhóm các từ theo chữ cái đầu
nhom = defaultdict(list)
for w in ["an", "bò", "ba", "ăn", "ai"]:
    nhom[w[0]].append(w)
print("Nhóm:", dict(nhom))

# 3) Ghi nhớ đã thấy: tìm hai chỉ số có tổng = target trong một lượt
def hai_so_tong(nums, target):
    vi_tri = {}                       # giá trị -> chỉ số
    for i, x in enumerate(nums):
        if target - x in vi_tri:      # phần bù đã gặp trước đó?
            return vi_tri[target - x], i
        vi_tri[x] = i
    return None
print("hai_so_tong([2, 7, 11, 15], 9) =", hai_so_tong([2, 7, 11, 15], 9))

# 4) Đã thăm
da_tham = set()
for v in [3, 1, 3, 2, 1]:
    if v in da_tham:
        print("gặp lại", v)
    da_tham.add(v)

Counter: Counter({'a': 3, 'n': 2, 'b': 1})
Nhóm: {'a': ['an', 'ai'], 'b': ['bò', 'ba'], 'ă': ['ăn']}
hai_so_tong([2, 7, 11, 15], 9) = (0, 1)
gặp lại 3
gặp lại 1


### Độ phức tạp và lưu ý

| Thao tác | `dict` / `set` (trung bình) | `list` |
|---|---|---|
| kiểm tra `x in ...` | `O(1)` | `O(n)` |
| thêm / xóa theo khóa | `O(1)` | xóa giữa: `O(n)` |
| duyệt toàn bộ | `O(n)` | `O(n)` |

Lỗi hay gặp:

- Dùng `list` làm khóa → `TypeError: unhashable type`. Đổi sang `tuple(...)`.
- Thêm/xóa khóa **trong lúc** đang duyệt chính dict đó → `RuntimeError`.
- Truy cập `d[k]` khi `k` chưa có → `KeyError`; dùng `d.get(k, mặc_định)` hoặc `defaultdict`.

In [4]:
try:
    {[1, 2]: "x"}
except TypeError as e:
    print("Khóa là list:", e)
print("Khóa là tuple thì được:", {(1, 2): "x"})

Khóa là list: unhashable type: 'list'
Khóa là tuple thì được: {(1, 2): 'x'}


## 2.2 Cửa sổ trượt (sliding window)

### Ý tưởng

Cửa sổ trượt là trường hợp riêng của hai con trỏ cùng chiều: hai đầu `left`, `right` xác định một **đoạn liên tiếp** `[left, right]`. Ta **mở rộng** bên phải từng bước; khi cửa sổ vi phạm điều kiện, **thu hẹp** bên trái cho tới khi hợp lệ lại. Mỗi phần tử vào cửa sổ một lần, ra một lần → `O(n)`.

```text
Đoạn con dài nhất có tổng <= 7 trong [2, 1, 5, 1, 3, 2]
[2]           tổng 2   ✓  dài 1
[2,1]         tổng 3   ✓  dài 2
[2,1,5]       tổng 8   ✗  → bỏ 2 → [1,5] tổng 6 ✓
[1,5,1]       tổng 7   ✓  dài 3
[1,5,1,3]     tổng 10  ✗  → bỏ 1, bỏ 5 → [1,3] tổng 4
[1,3,2]       tổng 6   ✓  dài 3
```

### Khung (cửa sổ thay đổi kích thước)

```text
left = 0
for right in range(n):
    thêm a[right] vào cửa sổ
    while cửa sổ không hợp lệ:
        bỏ a[left] khỏi cửa sổ; left += 1
    cập nhật đáp án với cửa sổ [left, right]
```

In [5]:
def dai_nhat_tong_khong_qua(a, k):
    left, tong, best = 0, 0, 0
    for right, x in enumerate(a):
        tong += x
        while tong > k:
            tong -= a[left]
            left += 1
        best = max(best, right - left + 1)
        print(f"   right={right} cửa sổ={a[left:right + 1]} tổng={tong}")
    return best

print("→ dài nhất:", dai_nhat_tong_khong_qua([2, 1, 5, 1, 3, 2], 7))

   right=0 cửa sổ=[2] tổng=2
   right=1 cửa sổ=[2, 1] tổng=3
   right=2 cửa sổ=[1, 5] tổng=6
   right=3 cửa sổ=[1, 5, 1] tổng=7
   right=4 cửa sổ=[1, 3] tổng=4
   right=5 cửa sổ=[1, 3, 2] tổng=6
→ dài nhất: 3


### Cửa sổ với bảng băm: đoạn không có ký tự lặp

Cửa sổ giữ **tập ký tự** bên trong. Khi ký tự mới đã có trong cửa sổ, thu hẹp trái cho tới khi bản cũ bị bỏ ra.

In [6]:
def dai_nhat_khong_lap(s):
    trong, left, best = set(), 0, 0
    for right, ch in enumerate(s):
        while ch in trong:
            trong.remove(s[left])
            left += 1
        trong.add(ch)
        best = max(best, right - left + 1)
    return best

for s_sw in ["abcabcbb", "bbbbb", "pwwkew", ""]:
    print(f"{s_sw!r:>11}: {dai_nhat_khong_lap(s_sw)}")

 'abcabcbb': 3
    'bbbbb': 1
   'pwwkew': 3
         '': 0


### Lỗi hay gặp

- Dùng `if` thay cho `while` khi thu hẹp → cửa sổ có thể vẫn chưa hợp lệ.
- Cập nhật đáp án **trước** khi làm cửa sổ hợp lệ.
- Áp dụng cho bài mà thu hẹp không giữ tính đơn điệu (ví dụ tổng với số **âm**: bỏ bớt phần tử chưa chắc làm tổng giảm).

## 2.3 Áp dụng vào bài này

Ta cần **đoạn con liên tiếp** dài nhất không có ký tự lặp → **cửa sổ trượt** `[left, right]`. Để biết ký tự mới đã có trong cửa sổ chưa trong `O(1)`, ta dùng **bảng băm**: `set` các ký tự trong cửa sổ, hoặc `dict` ký tự → vị trí xuất hiện gần nhất để nhảy `left` một lần thay vì thu hẹp từng bước.

# 3. Substring khác Subsequence

- **Substring**: một đoạn **liên tiếp** của chuỗi. Lấy bằng slicing `s[i:j]`.
- **Subsequence**: chọn một số ký tự, giữ thứ tự, **được phép bỏ qua** ký tự ở giữa.

Với `s = "pwwkew"`:

| Chuỗi | Substring? | Subsequence? |
|---|---|---|
| `"wke"` | Có (`s[2:5]`) | Có |
| `"pwke"` | **Không** | Có (bỏ chữ `w` thứ hai) |

## Nhắc lại slicing `s[i:j]`

`s[i:j]` lấy các ký tự từ chỉ số `i` **đến trước** `j` (không lấy `s[j]`). Độ dài của nó là `j - i`.

In [7]:
s = "pwwkew"
#    012345  <- chỉ số

print(s[2:5])        # 'wke'  (chỉ số 2, 3, 4)
print(len(s[2:5]))   # 3 = 5 - 2
print(s[0:2])        # 'pw'
print(s[3:3])        # ''  (chuỗi rỗng khi i == j)
print("pwke" in s)   # False: 'pwke' không phải substring

wke
3
pw

False


# 4. Công cụ kiểm tra trùng: `set`

`set` (tập hợp) là cấu trúc chỉ chứa **các phần tử khác nhau**. Các thao tác ta cần:

- `seen = set()` — tạo tập rỗng.
- `seen.add(c)` — thêm phần tử.
- `c in seen` — kiểm tra tồn tại, trung bình `O(1)`.
- `seen.remove(c)` — xóa phần tử (phải chắc nó đang có trong set).

Mẹo: một chuỗi **không có ký tự lặp** khi và chỉ khi `len(set(t)) == len(t)`.

In [8]:
seen = set()
seen.add("a")
seen.add("b")
seen.add("a")          # thêm trùng -> không thay đổi gì
print(seen, len(seen))
print("a" in seen, "z" in seen)

seen.remove("a")
print(seen)

for t in ["abc", "abca", "bbbbb", ""]:
    # sorted(...) chỉ để in ra theo thứ tự cố định; set vốn không có thứ tự
    print(f"{t!r:8} -> các ký tự khác nhau = {sorted(set(t))!s:17} không lặp? {len(set(t)) == len(t)}")

{'b', 'a'} 2
True False
{'b'}
'abc'    -> các ký tự khác nhau = ['a', 'b', 'c']   không lặp? True
'abca'   -> các ký tự khác nhau = ['a', 'b', 'c']   không lặp? False
'bbbbb'  -> các ký tự khác nhau = ['b']             không lặp? False
''       -> các ký tự khác nhau = []                không lặp? True


# 5. Cách dễ nghĩ nhất: thử mọi chuỗi con

Duyệt mọi cặp `(i, j)` với `i < j`, lấy `t = s[i:j]`, nếu `t` không lặp thì cập nhật đáp án.

Với chuỗi dài `n` có khoảng `n * (n + 1) / 2` chuỗi con, mỗi chuỗi con mất tới `O(n)` để tạo và kiểm tra.

In [9]:
def length_bruteforce(s: str) -> int:
    n = len(s)
    best = 0
    for i in range(n):
        for j in range(i + 1, n + 1):
            t = s[i:j]
            if len(set(t)) == len(t):
                best = max(best, len(t))
    return best

print(length_bruteforce("abcabcbb"))  # 3
print(length_bruteforce("bbbbb"))     # 1
print(length_bruteforce("pwwkew"))    # 3
print(length_bruteforce(""))          # 0

3
1
3
0


## Độ phức tạp của Brute Force

```text
Số chuỗi con: ~ n^2 / 2
Mỗi chuỗi con: O(n) để cắt + tạo set
=> Tổng: O(n^3)
```

Với `n = 50 000`, `n^3` vào khoảng `1.25 * 10^14` phép tính — quá chậm.

# 6. Cải tiến lần 1: dừng ngay khi gặp ký tự lặp

Quan sát: nếu `s[i:j]` đã có ký tự lặp, thì mọi chuỗi dài hơn `s[i:j+1]`, `s[i:j+2]`, ... cũng lặp. Không cần thử tiếp.

Vì vậy, với mỗi điểm bắt đầu `i`, ta **kéo dài** sang phải và giữ một `set` các ký tự đã gặp. Gặp ký tự trùng thì dừng.

In [10]:
def length_extend_each_start(s: str) -> int:
    n = len(s)
    best = 0
    for i in range(n):
        seen = set()
        for j in range(i, n):
            if s[j] in seen:
                break
            seen.add(s[j])
            best = max(best, j - i + 1)
    return best

print(length_extend_each_start("abcabcbb"))
print(length_extend_each_start("pwwkew"))

3
3


## Vẫn còn việc thừa ở đâu?

Xem `s = "abcabcbb"`:

- Bắt đầu tại `i = 0`: đi qua `a, b, c`, gặp `a` ở `j = 3` thì dừng. Ta biết `"abc"` không lặp.
- Bắt đầu tại `i = 1`: lại kiểm tra `b, c` **một lần nữa**, rồi mới tới `a`.

Nhưng ta đã biết `"abc"` không lặp, thì `"bc"` (bỏ bớt chữ đầu) **chắc chắn** cũng không lặp!

> Không cần bắt đầu lại từ đầu. Chỉ cần **bỏ ký tự bên trái** và **giữ nguyên phần còn lại**.

Đó chính là ý tưởng **cửa sổ trượt**.

# 7. Ý tưởng Sliding Window

Ta giữ một "cửa sổ" là đoạn `s[left : right + 1]`, với **bất biến**:

> Cửa sổ luôn **không chứa ký tự lặp**.

Mỗi bước:

1. Mở rộng: đưa `s[right]` vào cửa sổ.
2. Nếu `s[right]` đã có trong cửa sổ → **thu hẹp từ trái**: bỏ `s[left]`, tăng `left`, lặp lại cho tới khi hết trùng.
3. Lúc này cửa sổ hợp lệ, độ dài là `right - left + 1`. Cập nhật đáp án.

```text
s = a b c a b c b b
    L     R            cửa sổ "abc" hợp lệ
    L       R          thêm 'a' -> trùng!
      L     R          bỏ 'a' bên trái -> "bca" hợp lệ
```

Cả `left` và `right` **chỉ đi về bên phải**, không bao giờ lùi lại.

In [11]:
# Độ dài cửa sổ s[left : right + 1]
s = "abcabcbb"
left, right = 1, 3
print("cửa sổ =", s[left:right + 1])
print("độ dài =", right - left + 1)

cửa sổ = bca
độ dài = 3


# 8. Lời giải Sliding Window với `set`

`window` là `set` chứa các ký tự đang nằm trong cửa sổ.

Chú ý: phải dùng **`while`**, không phải `if`, khi thu hẹp — có thể phải bỏ **nhiều** ký tự bên trái mới hết trùng.

In [12]:
def length_sliding_set(s: str) -> int:
    window = set()
    left = 0
    best = 0

    for right, c in enumerate(s):
        while c in window:          # còn trùng thì thu hẹp
            window.remove(s[left])
            left += 1

        window.add(c)
        best = max(best, right - left + 1)

    return best

print(length_sliding_set("abcabcbb"))  # 3
print(length_sliding_set("bbbbb"))     # 1
print(length_sliding_set("pwwkew"))    # 3
print(length_sliding_set(""))          # 0

3
1
3
0


# 9. Mô phỏng từng bước

Xét `s = "pwwkew"` (đáp án 3).

| right | c | Trùng? | Thu hẹp | left sau | Cửa sổ | Độ dài | best |
|---|---|---|---|---|---|---|---|
| 0 | p | không | — | 0 | `p` | 1 | 1 |
| 1 | w | không | — | 0 | `pw` | 2 | 2 |
| 2 | w | **có** | bỏ `p`, bỏ `w` | 2 | `w` | 1 | 2 |
| 3 | k | không | — | 2 | `wk` | 2 | 2 |
| 4 | e | không | — | 2 | `wke` | 3 | 3 |
| 5 | w | **có** | bỏ `w` | 3 | `kew` | 3 | 3 |

Ở `right = 2`, ta phải bỏ **hai** ký tự (`p` rồi `w`) — đây là lý do cần `while`.

In [13]:
def trace_sliding_set(s: str) -> int:
    window = set()
    left = 0
    best = 0

    for right, c in enumerate(s):
        print("=" * 50)
        print(f"right = {right}, c = {c!r}, cửa sổ trước = {s[left:right]!r}")

        while c in window:
            print(f"  {c!r} đã có -> bỏ s[{left}] = {s[left]!r}")
            window.remove(s[left])
            left += 1

        window.add(c)
        length = right - left + 1
        best = max(best, length)
        print(f"  cửa sổ sau = {s[left:right + 1]!r} (left={left}), độ dài = {length}, best = {best}")

    return best

print("Đáp án:", trace_sliding_set("pwwkew"))

right = 0, c = 'p', cửa sổ trước = ''
  cửa sổ sau = 'p' (left=0), độ dài = 1, best = 1
right = 1, c = 'w', cửa sổ trước = 'p'


  cửa sổ sau = 'pw' (left=0), độ dài = 2, best = 2
right = 2, c = 'w', cửa sổ trước = 'pw'
  'w' đã có -> bỏ s[0] = 'p'
  'w' đã có -> bỏ s[1] = 'w'
  cửa sổ sau = 'w' (left=2), độ dài = 1, best = 2
right = 3, c = 'k', cửa sổ trước = 'w'
  cửa sổ sau = 'wk' (left=2), độ dài = 2, best = 2
right = 4, c = 'e', cửa sổ trước = 'wk'
  cửa sổ sau = 'wke' (left=2), độ dài = 3, best = 3
right = 5, c = 'w', cửa sổ trước = 'wke'
  'w' đã có -> bỏ s[2] = 'w'
  cửa sổ sau = 'kew' (left=3), độ dài = 3, best = 3
Đáp án: 3


# 10. Tối ưu thêm: nhảy `left` bằng `dict`

Với `set`, khi gặp trùng ta phải bỏ từng ký tự một. Nếu biết **ký tự `c` xuất hiện lần trước ở đâu**, ta có thể **nhảy thẳng** `left` qua vị trí đó.

Dùng `last = {}` lưu `ký tự -> chỉ số xuất hiện gần nhất`.

Khi gặp `c` tại `right`:

```python
if c in last and last[c] >= left:   # lần xuất hiện trước nằm TRONG cửa sổ
    left = last[c] + 1
last[c] = right
```

Có thể viết gọn bằng `max`:

```python
if c in last:
    left = max(left, last[c] + 1)
```

## Vì sao cần `max`?

`last[c]` có thể nằm **bên trái cửa sổ** (đã bị bỏ đi từ trước). Khi đó **không được** kéo `left` lùi lại.

Ví dụ `s = "abba"`:

| right | c | last trước | left trước | last[c] + 1 | left sau | Cửa sổ |
|---|---|---|---|---|---|---|
| 0 | a | {} | 0 | — | 0 | `a` |
| 1 | b | {a:0} | 0 | — | 0 | `ab` |
| 2 | b | {a:0, b:1} | 0 | 2 | 2 | `b` |
| 3 | a | {a:0, b:2} | 2 | **1** | **2** (giữ nguyên) | `ba` |

Ở `right = 3`: `last['a'] = 0` nằm ngoài cửa sổ. Nếu không dùng `max`, `left` sẽ lùi về `1`, cửa sổ thành `"bba"` — có lặp!

In [14]:
def length_sliding_dict(s: str) -> int:
    last = {}
    left = 0
    best = 0

    for right, c in enumerate(s):
        if c in last:
            left = max(left, last[c] + 1)
        last[c] = right
        best = max(best, right - left + 1)

    return best

for s in ["abcabcbb", "bbbbb", "pwwkew", "", "abba", "dvdf"]:
    print(f"{s!r:12} -> {length_sliding_dict(s)}")

'abcabcbb'   -> 3
'bbbbb'      -> 1
'pwwkew'     -> 3
''           -> 0
'abba'       -> 2
'dvdf'       -> 3


In [15]:
def trace_sliding_dict(s: str) -> int:
    last = {}
    left = 0
    best = 0
    for right, c in enumerate(s):
        old_left = left
        if c in last:
            left = max(left, last[c] + 1)
        note = "" if left == old_left else f"  (nhảy left {old_left} -> {left})"
        last[c] = right
        best = max(best, right - left + 1)
        print(f"right={right} c={c!r} cửa sổ={s[left:right + 1]!r:8} best={best}{note}  last={last}")
    return best

print("Đáp án:", trace_sliding_dict("abba"))

right=0 c='a' cửa sổ='a'      best=1  last={'a': 0}
right=1 c='b' cửa sổ='ab'     best=2  last={'a': 0, 'b': 1}
right=2 c='b' cửa sổ='b'      best=2  (nhảy left 0 -> 2)  last={'a': 0, 'b': 2}
right=3 c='a' cửa sổ='ba'     best=2  last={'a': 3, 'b': 2}
Đáp án: 2


# 11. Phiên bản đúng format LeetCode

Ta nộp bản `dict` (gọn và chỉ duyệt một lần). Bản `set` cũng được chấp nhận.

In [16]:
class Solution:
    def lengthOfLongestSubstring(self, s: str) -> int:
        last = {}
        left = 0
        best = 0

        for right, c in enumerate(s):
            if c in last:
                left = max(left, last[c] + 1)
            last[c] = right
            best = max(best, right - left + 1)

        return best

In [17]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<30} PASS  {actual}")


sol = Solution()
check("Ví dụ 1: abcabcbb", sol.lengthOfLongestSubstring("abcabcbb"), 3)
check("Ví dụ 2: bbbbb", sol.lengthOfLongestSubstring("bbbbb"), 1)
check("Ví dụ 3: pwwkew", sol.lengthOfLongestSubstring("pwwkew"), 3)
check("Chuỗi rỗng", sol.lengthOfLongestSubstring(""), 0)
check("Một ký tự", sol.lengthOfLongestSubstring("x"), 1)
check("Dấu cách", sol.lengthOfLongestSubstring(" "), 1)
check("abba (cần max)", sol.lengthOfLongestSubstring("abba"), 2)
check("dvdf", sol.lengthOfLongestSubstring("dvdf"), 3)
check("Toàn khác nhau", sol.lengthOfLongestSubstring("abcdef"), 6)

Ví dụ 1: abcabcbb              PASS  3
Ví dụ 2: bbbbb                 PASS  1
Ví dụ 3: pwwkew                PASS  3
Chuỗi rỗng                     PASS  0
Một ký tự                      PASS  1
Dấu cách                       PASS  1
abba (cần max)                 PASS  2
dvdf                           PASS  3
Toàn khác nhau                 PASS  6


## Đối chiếu nhiều cách giải

Sinh chuỗi ngẫu nhiên từ một bảng chữ nhỏ (để dễ có ký tự lặp) và so sánh cả 4 cách.

In [18]:
import random

random.seed(1)
alphabet = "abcd "

for _ in range(2000):
    s = "".join(random.choice(alphabet) for _ in range(random.randint(0, 12)))
    expected = length_bruteforce(s)
    assert length_extend_each_start(s) == expected, s
    assert length_sliding_set(s) == expected, s
    assert sol.lengthOfLongestSubstring(s) == expected, s

print("2000 test ngẫu nhiên: 4 cách cho cùng kết quả")

2000 test ngẫu nhiên: 4 cách cho cùng kết quả


# 12. Vì sao thuật toán đúng?

**Bất biến:** sau khi xử lý `right`, cửa sổ `s[left : right + 1]` là chuỗi con **dài nhất không lặp kết thúc tại `right`**.

- Không lặp: mỗi lần thêm `c`, ta đẩy `left` qua lần xuất hiện trước của `c` (nếu nó còn trong cửa sổ). Các ký tự khác vốn không lặp từ bước trước.
- Dài nhất: `left` chỉ dịch **vừa đủ** để bỏ ký tự trùng. Nếu `left` nhỏ hơn một chút, cửa sổ sẽ chứa hai chữ `c`.

Chuỗi con tốt nhất của cả bài phải **kết thúc ở một vị trí `right` nào đó**. Ta đã xét mọi `right` và lấy `max`, nên không bỏ sót.

# 13. Độ phức tạp

Gọi `n = len(s)` và `k` là số ký tự khác nhau có thể xuất hiện (bảng chữ).

**Bản `set`:** `right` đi `n` bước. `left` cũng chỉ tăng, tổng cộng tối đa `n` bước. Mỗi ký tự được `add` một lần và `remove` tối đa một lần → **O(n)** tổng cộng (dù có `while` lồng trong `for`).

**Bản `dict`:** mỗi `right` làm vài thao tác `O(1)` trung bình → **O(n)**.

```text
Time complexity:  O(n)
Space complexity: O(min(n, k))   (set/dict chỉ chứa các ký tự khác nhau)
```

| Cách làm | Thời gian | Bộ nhớ |
|---|---|---|
| Thử mọi chuỗi con | O(n³) | O(n) |
| Kéo dài từ mỗi điểm đầu | O(n · min(n, k)) | O(min(n, k)) |
| Sliding window + set | O(n) | O(min(n, k)) |
| Sliding window + dict | O(n) | O(min(n, k)) |

> Một `while` nằm trong `for` **không** tự động là `O(n²)`. Hãy đếm tổng số lần con trỏ di chuyển.

# 14. Những lỗi dễ mắc

## Lỗi 1: Dùng `if` thay vì `while` khi thu hẹp (bản `set`)

Chỉ bỏ **một** ký tự bên trái là chưa đủ. Với `"pwwkew"`, sau khi bỏ `p`, cửa sổ vẫn còn `w`.

In [19]:
def wrong_if_instead_of_while(s: str) -> int:
    window = set()
    left = 0
    best = 0
    for right, c in enumerate(s):
        if c in window:             # SAI: chỉ bỏ 1 ký tự
            window.remove(s[left])
            left += 1
        window.add(c)
        best = max(best, right - left + 1)
    return best

print("Sai:", wrong_if_instead_of_while("pwwkew"))
print("Đúng:", length_sliding_set("pwwkew"))

Sai: 4
Đúng: 3


## Lỗi 2: Quên `max` khi nhảy `left` (bản `dict`)

In [20]:
def wrong_no_max(s: str) -> int:
    last = {}
    left = 0
    best = 0
    for right, c in enumerate(s):
        if c in last:
            left = last[c] + 1      # SAI: có thể kéo left lùi lại
        last[c] = right
        best = max(best, right - left + 1)
    return best

print("Sai:", wrong_no_max("abba"))
print("Đúng:", length_sliding_dict("abba"))

Sai: 3
Đúng: 2


## Lỗi 3: Tính sai độ dài cửa sổ

Cửa sổ là `s[left : right + 1]`, độ dài là `right - left + 1`, **không phải** `right - left`.

Với `s = "a"`: `right = 0`, `left = 0` → độ dài đúng là `1`.

## Lỗi 4: Nhầm substring với subsequence

`"pwwkew"` không có đáp án 4 (`"pwke"`) vì substring phải liên tiếp.

## Lỗi 5: Quên chuỗi rỗng

`s = ""` hợp lệ theo constraints. Vòng `for` không chạy lần nào và trả về `best = 0` — đúng. Nhưng nếu bạn viết `s[0]` ở đâu đó trước vòng lặp sẽ bị `IndexError`.

# 15. Mẹo nhận diện bài Sliding Window

Nghĩ tới cửa sổ trượt khi đề có các dấu hiệu:

- "**substring / subarray** (liên tiếp) **dài nhất / ngắn nhất**…"
- "…sao cho **thỏa một điều kiện**" (không lặp, tổng ≤ k, có tối đa k ký tự khác nhau…)
- Điều kiện có tính **đơn điệu**: nếu một đoạn vi phạm, thì mở rộng đoạn đó vẫn vi phạm → có thể thu hẹp từ trái.

## Template cửa sổ thay đổi kích thước

```python
left = 0
best = 0
state = ...                  # set / dict / tổng / bộ đếm

for right in range(len(s)):
    thêm s[right] vào state

    while cửa sổ vi phạm điều kiện:
        bỏ s[left] khỏi state
        left += 1

    best = max(best, right - left + 1)
```

Các bài cùng họ: **#159 / #340** (tối đa 2 / k ký tự khác nhau), **#424** (thay thế ký tự để có chuỗi lặp dài nhất), **#209** (mảng con tổng ≥ target ngắn nhất), **#76** (Minimum Window Substring).

# 16. Biến thể 1: Trả về chính chuỗi con

Chỉ cần nhớ thêm vị trí bắt đầu `best_start` mỗi khi cập nhật `best`.

In [21]:
def longest_unique_substring(s: str) -> str:
    last = {}
    left = 0
    best = 0
    best_start = 0

    for right, c in enumerate(s):
        if c in last:
            left = max(left, last[c] + 1)
        last[c] = right
        if right - left + 1 > best:
            best = right - left + 1
            best_start = left

    return s[best_start:best_start + best]

print(repr(longest_unique_substring("abcabcbb")))
print(repr(longest_unique_substring("pwwkew")))
print(repr(longest_unique_substring("")))

'abc'
'wke'
''


# 17. Biến thể 2: Cửa sổ kích thước cố định

Không phải bài Sliding Window nào cũng co giãn. Ví dụ: **tổng lớn nhất của `k` phần tử liên tiếp**.

Ý tưởng: tính tổng `k` phần tử đầu, rồi mỗi bước **cộng phần tử mới vào, trừ phần tử rơi ra**.

In [22]:
def max_sum_window(nums, k):
    window_sum = sum(nums[:k])
    best = window_sum
    for right in range(k, len(nums)):
        window_sum += nums[right] - nums[right - k]
        best = max(best, window_sum)
    return best

print(max_sum_window([2, 1, 5, 1, 3, 2], 3))   # 5 + 1 + 3 = 9
print(max_sum_window([1, -2, 3, 10, -4], 2))   # 3 + 10 = 13

9


13


# 18. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Không chạy code. Lập bảng trace bản `dict` cho `s = "dvdf"` với các cột `right`, `c`, `last trước`, `left`, cửa sổ, `best`.

Đáp án cuối là bao nhiêu? Ở bước nào `left` nhảy?

## Bài 2 — Đoán output

```python
s = "tmmzuxt"
print(s[1:4])
print(len(set(s)))
print(len(set(s)) == len(s))
print(s[2:2] == "")
```

## Bài 3 — Chuỗi có ký tự lặp không?

Viết hàm trả về `True` nếu `s` có ít nhất một ký tự xuất hiện 2 lần. Yêu cầu: duyệt một lần, dừng sớm khi tìm thấy.

```python
has_repeat("abc")   -> False
has_repeat("abca")  -> True
has_repeat("")      -> False
```

In [23]:
def exercise_3_has_repeat(s: str) -> bool:
    # TODO:
    # Tạo set seen
    # Duyệt từng ký tự: nếu đã có trong seen -> True, ngược lại thêm vào
    pass


# assert exercise_3_has_repeat("abc") is False
# assert exercise_3_has_repeat("abca") is True
# assert exercise_3_has_repeat("") is False

## Bài 4 — Điền chỗ trống: Sliding Window với `set`

In [24]:
def exercise_4_length_sliding_set(s: str) -> int:
    window = set()
    left = 0
    best = 0

    for right, c in enumerate(s):
        # TODO 1: trong khi c còn trong window, bỏ s[left] và tăng left
        # TODO 2: thêm c vào window
        # TODO 3: cập nhật best bằng độ dài cửa sổ hiện tại
        pass

    return best


# assert exercise_4_length_sliding_set("abcabcbb") == 3
# assert exercise_4_length_sliding_set("pwwkew") == 3
# assert exercise_4_length_sliding_set("abba") == 2

## Bài 5 — Biến thể: tối đa 2 ký tự khác nhau (LeetCode #159)

Tìm độ dài chuỗi con dài nhất chứa **tối đa 2 ký tự khác nhau**.

```python
at_most_two("eceba")    -> 3   ("ece")
at_most_two("ccaabbb")  -> 5   ("aabbb")
```

Gợi ý nhanh: `set` không đủ vì cần biết **số lần** mỗi ký tự còn trong cửa sổ.

In [25]:
def exercise_5_at_most_two(s: str) -> int:
    # TODO
    pass


# assert exercise_5_at_most_two("eceba") == 3
# assert exercise_5_at_most_two("ccaabbb") == 5
# assert exercise_5_at_most_two("") == 0

## Bài 6 — Mini challenge: tối đa `k` ký tự khác nhau (LeetCode #340)

Tổng quát bài 5 cho `k` bất kỳ (`k >= 0`).

```python
at_most_k("eceba", 2)  -> 3
at_most_k("aa", 1)     -> 2
at_most_k("abc", 0)    -> 0
```

Sau khi làm xong, hãy suy nghĩ: bài gốc #3 có thể xem là trường hợp "mọi ký tự trong cửa sổ có số lần xuất hiện ≤ 1". Bạn có viết lại bài #3 bằng bộ đếm được không?

In [26]:
def exercise_6_at_most_k(s: str, k: int) -> int:
    # TODO
    pass


# assert exercise_6_at_most_k("eceba", 2) == 3
# assert exercise_6_at_most_k("aa", 1) == 2
# assert exercise_6_at_most_k("abc", 0) == 0

# 19. Gợi ý

**Bài 1**
- Mức 1: nhớ `left = max(left, last[c] + 1)`.
- Mức 2: lần duy nhất gặp lại ký tự là chữ `d` thứ hai.
- Mức 3: khi đó `last['d'] = 0`, nên `left` thành `1`.

**Bài 3**
- Mức 1: giống Contains Duplicate nhưng trên chuỗi.
- Mức 2: `if c in seen: return True`.
- Mức 3: sau vòng lặp `return False`.

**Bài 4**
- Mức 1: so với code mục 8.
- Mức 2: dùng `while`, không phải `if`.
- Mức 3: độ dài là `right - left + 1`.

**Bài 5 / Bài 6**
- Mức 1: cửa sổ vi phạm khi số ký tự khác nhau `> k`.
- Mức 2: dùng `count = {}` đếm số lần mỗi ký tự trong cửa sổ; số ký tự khác nhau là `len(count)`.
- Mức 3: khi thu hẹp, giảm `count[s[left]]`; nếu về `0` thì `del count[s[left]]`.

# 20. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

| right | c | last trước | left | Cửa sổ | best |
|---|---|---|---|---|---|
| 0 | d | {} | 0 | `d` | 1 |
| 1 | v | {d:0} | 0 | `dv` | 2 |
| 2 | d | {d:0, v:1} | **1** (nhảy) | `vd` | 2 |
| 3 | f | {d:2, v:1} | 1 | `vdf` | 3 |

Đáp án `3`. `left` nhảy ở `right = 2`.

Lưu ý: nếu bạn "khởi động lại" từ vị trí trùng thứ hai (`left = 2`) thì sẽ ra `2` — sai. Phải nhảy tới **sau lần xuất hiện trước**, tức `0 + 1 = 1`.

In [27]:
trace_sliding_dict("dvdf")

right=0 c='d' cửa sổ='d'      best=1  last={'d': 0}
right=1 c='v' cửa sổ='dv'     best=2  last={'d': 0, 'v': 1}
right=2 c='d' cửa sổ='vd'     best=2  (nhảy left 0 -> 1)  last={'d': 2, 'v': 1}
right=3 c='f' cửa sổ='vdf'    best=3  last={'d': 2, 'v': 1, 'f': 3}


3

## Đáp án Bài 2

In [28]:
s = "tmmzuxt"
print(s[1:4])                 # 'mmz'
print(len(set(s)))            # 5: t, m, z, u, x
print(len(set(s)) == len(s))  # False (có lặp)
print(s[2:2] == "")           # True

mmz
5
False
True


## Đáp án Bài 3

In [29]:
def answer_3_has_repeat(s: str) -> bool:
    seen = set()
    for c in s:
        if c in seen:
            return True
        seen.add(c)
    return False

assert answer_3_has_repeat("abc") is False
assert answer_3_has_repeat("abca") is True
assert answer_3_has_repeat("") is False
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [30]:
def answer_4_length_sliding_set(s: str) -> int:
    window = set()
    left = 0
    best = 0
    for right, c in enumerate(s):
        while c in window:
            window.remove(s[left])
            left += 1
        window.add(c)
        best = max(best, right - left + 1)
    return best

assert answer_4_length_sliding_set("abcabcbb") == 3
assert answer_4_length_sliding_set("pwwkew") == 3
assert answer_4_length_sliding_set("abba") == 2
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5 và Bài 6

Bài 5 chính là Bài 6 với `k = 2`.

In [31]:
def answer_6_at_most_k(s: str, k: int) -> int:
    count = {}
    left = 0
    best = 0

    for right, c in enumerate(s):
        count[c] = count.get(c, 0) + 1

        while len(count) > k:
            out = s[left]
            count[out] -= 1
            if count[out] == 0:
                del count[out]
            left += 1

        best = max(best, right - left + 1)

    return best


def answer_5_at_most_two(s: str) -> int:
    return answer_6_at_most_k(s, 2)


assert answer_5_at_most_two("eceba") == 3
assert answer_5_at_most_two("ccaabbb") == 5
assert answer_5_at_most_two("") == 0
assert answer_6_at_most_k("eceba", 2) == 3
assert answer_6_at_most_k("aa", 1) == 2
assert answer_6_at_most_k("abc", 0) == 0
print("Bài 5, 6 PASS")

Bài 5, 6 PASS


### Câu hỏi mở của Bài 6: viết lại bài #3 bằng bộ đếm

Cửa sổ vi phạm khi ký tự vừa thêm có số lần `> 1`.

In [32]:
def length_with_counter(s: str) -> int:
    count = {}
    left = 0
    best = 0
    for right, c in enumerate(s):
        count[c] = count.get(c, 0) + 1
        while count[c] > 1:
            count[s[left]] -= 1
            left += 1
        best = max(best, right - left + 1)
    return best

for s in ["abcabcbb", "bbbbb", "pwwkew", "abba", ""]:
    assert length_with_counter(s) == length_bruteforce(s)
print("Bản bộ đếm PASS")

Bản bộ đếm PASS


# 21. Tổng kết

Bài #3 là bài nhập môn kinh điển cho **Sliding Window**.

Cốt lõi:

> Giữ một cửa sổ luôn hợp lệ. Mở rộng bên phải từng bước; khi vi phạm thì thu hẹp bên trái. Cả hai con trỏ chỉ đi tới, nên tổng chi phí là `O(n)`.

## Công thức nhớ nhanh

```python
last = {}
left = 0
best = 0
for right, c in enumerate(s):
    if c in last:
        left = max(left, last[c] + 1)
    last[c] = right
    best = max(best, right - left + 1)
return best
```

---

## Checklist trước khi nộp

- [ ] Trả về **độ dài**, không phải chuỗi?
- [ ] Bản `set`: dùng `while` khi thu hẹp?
- [ ] Bản `dict`: có `max(left, last[c] + 1)`?
- [ ] Độ dài cửa sổ là `right - left + 1`?
- [ ] Đã thử `""`, `"abba"`, `"dvdf"`, `"bbbbb"`?

---

## Khi nào nên nghĩ tới Sliding Window?

- "substring / subarray **liên tiếp**"
- "**dài nhất / ngắn nhất** thỏa điều kiện"
- Có thể kiểm tra điều kiện của cửa sổ **tăng dần** khi thêm/bớt một phần tử
- Brute force đang thử mọi cặp `(i, j)` và lặp lại nhiều việc giống nhau